# Lab L4.1 · SVMs in production conditions — tuned RBF fraud classifier, kernel approximation at scale, and SVR for loss severity
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 4 · ⏱ 70 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Tune C and γ properly, hit the wall of kernel-SVM complexity, break through it with a Nyström approximation, and apply SVR to predict loss amounts on defaulted loans.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

A vendor claims their *SVM fraud engine* beats HGB’s random forest. You will run a fair bake-off on HGB’s own features and time budget. Separately, the IFRS 9 team needs a **loss-severity** model for loans that do default — a regression problem where SVR is a candidate.

**Progressive architecture:** reuses L3.3’s feature table and benchmark, and L1.2’s loan features.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load L3.3 features and define the evaluation

**Why this matters.** A bake-off is only fair if every model sees the same features, the same split and the same metric. We reuse L3.3’s table and its `at_budget` metric, and record the RF benchmark.

In [ ]:
import json
cf = pd.read_parquet(CKPT / "card_features.parquet")                  # from Lab L3.3
fm = json.loads((CKPT / "fraud_metrics.json").read_text())
NUMF = [f for f in fm["features"] if f not in ("merchant_category", "channel")]
CATF = ["merchant_category", "channel"]
tr, va, te = (cf[cf.split == s] for s in ["train", "valid", "test"])
print(f"{len(cf):,} transactions | RF benchmark from L3.3 (valid): PR-AUC {fm['valid']['PR_AUC']:.3f}, recall@0.5% {fm['valid']['recall@0.5%']:.3f}")

In [ ]:
from sklearn.metrics import average_precision_score
def at_budget(y, s, amount, budget=0.005) -> dict:
    """Ranking quality inside the fraud team's review budget (s = any score; higher = riskier)."""
    y, amount = np.asarray(y), np.asarray(amount)
    top = np.argsort(-np.asarray(s))[: int(len(y) * budget)]
    return {"precision@0.5%": y[top].mean(), "recall@0.5%": y[top].sum() / y.sum(),
            "AED_caught_share": (amount[top] * y[top]).sum() / (amount * y).sum(), "PR_AUC": average_precision_score(y, s)}

**✔ Checkpoint.** 1.5M rows; RF benchmark PR-AUC ≈ 0.94, recall@0.5% ≈ 0.89.

## Step 2 · Preprocessing for SVMs and a LinearSVC baseline

**Why this matters.** SVMs need **numeric, scaled, complete** inputs: impute, one-hot the two categorical columns (an ordinal code would impose a fake distance) and standardise. A **LinearSVC** on the undersampled training set is the cheap baseline.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — SVMs are distance-based: scale is mandatory
- **Blank 2** *(inline)* — no fake ordering between categories
- **Blank 3** *(inline)* — signed distance to the hyperplane = risk score

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import LinearSVC, SVC, SVR
svm_prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="constant", fill_value=-1), ____), NUMF),  # ✏️ B1
    ("cat", ____, CATF)])  # ✏️ B2
FEATS = NUMF + CATF
tr_u = pd.concat([tr[tr.is_fraud == 1], tr[tr.is_fraud == 0].sample(30_000, random_state=RNG)]).sample(frac=1, random_state=RNG)
board, timing = {}, {}
t0 = time.perf_counter()
lin = Pipeline([("prep", svm_prep), ("svm", LinearSVC(C=0.1, class_weight="balanced"))]).fit(tr_u[FEATS], tr_u.is_fraud)
timing["LinearSVC fit (s)"] = time.perf_counter() - t0
board["LinearSVC"] = at_budget(va.is_fraud, ____, va.amount_aed)  # ✏️ B3
print(len(tr_u), timing); pd.DataFrame(board).T.round(3)

**✔ Checkpoint.** Fits in well under a second; recall@0.5% ≈ 0.88, PR-AUC ≈ 0.93 — already competitive because the engineered features are strong.

## Step 3 · Tune an RBF SVM with cross-validation on a stratified sample

**Why this matters.** An RBF SVM can model interactions a linear SVM cannot — but only with the right C and γ. Grid-searching on all 34k rows would take too long, so we tune on a **stratified 6k sample** with 3-fold CV and PR-AUC scoring, then refit the winner on the full undersampled set.

#### ✏️ What to write in the next cell
- **Blank 4** *(inline)* — margin-violation penalty, log scale
- **Blank 5** *(inline)* — RBF reach, log scale

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split
tune, _ = train_test_split(tr_u, train_size=6_000, stratify=tr_u.is_fraud, random_state=RNG)
grid = {"svm__C": ____, "svm__gamma": ____}  # ✏️ B4; B5
gs = GridSearchCV(Pipeline([("prep", svm_prep), ("svm", SVC(kernel="rbf"))]), grid, scoring="average_precision", cv=3, n_jobs=-1).fit(tune[FEATS], tune.is_fraud)
heat = pd.DataFrame(gs.cv_results_).pivot_table(index="param_svm__C", columns="param_svm__gamma", values="mean_test_score")
display(heat.style.background_gradient(cmap="Greens").format("{:.3f}")); print("best:", gs.best_params_)

In [ ]:
best = gs.best_params_
t0 = time.perf_counter()
rbf = Pipeline([("prep", svm_prep), ("svm", SVC(kernel="rbf", C=best["svm__C"], gamma=best["svm__gamma"]))]).fit(tr_u[FEATS], tr_u.is_fraud)
timing["RBF SVC fit (s)"] = time.perf_counter() - t0
t0 = time.perf_counter(); s_rbf = rbf.decision_function(va[FEATS]); timing["RBF SVC score valid (s)"] = time.perf_counter() - t0
board["RBF SVC (tuned)"] = at_budget(va.is_fraud, s_rbf, va.amount_aed)
print(f"support vectors: {rbf.named_steps['svm'].n_support_.sum():,}", {k: round(v, 1) for k, v in timing.items()})
pd.DataFrame(board).T.round(3)

**✔ Checkpoint.** Small γ (0.01–0.03) wins; large γ over-fits. The tuned RBF SVM edges past LinearSVC (PR-AUC ≈ 0.94 vs 0.93). Training is quick at 34k rows (≈ 1 s, ≈ 1,000 support vectors) but **scoring** 255k transactions takes ≈ 10–15 s — 100× slower than LinearSVC.

## Step 4 · Kernel approximation: RBF-like power at linear cost

**Why this matters.** **Nyström** samples landmark points and builds an explicit feature map that approximates the RBF kernel. A linear SVM on those features behaves like a kernel SVM but trains in roughly linear time — so we can use **all** training fraud and many more legitimate rows.

#### ✏️ What to write in the next cell
- **Blank 6** *(inline)* — explicit approximate RBF feature map

In [ ]:
from sklearn.kernel_approximation import Nystroem
tr_big = pd.concat([tr[tr.is_fraud == 1], tr[tr.is_fraud == 0].sample(250_000, random_state=RNG)])
t0 = time.perf_counter()
nys = Pipeline([("prep", svm_prep),
                ("map", ____),  # ✏️ B6
                ("svm", LinearSVC(C=0.5, class_weight="balanced"))]).fit(tr_big[FEATS], tr_big.is_fraud)
timing["Nystroem+LinearSVC fit on 254k (s)"] = time.perf_counter() - t0
board["Nystroem + LinearSVC (254k rows)"] = at_budget(va.is_fraud, nys.decision_function(va[FEATS]), va.amount_aed)
board["Random forest (L3.3 benchmark)"] = {k: fm["valid"][k] for k in ["precision@0.5%", "recall@0.5%", "AED_caught_share", "PR_AUC"]}
print({k: round(v, 1) for k, v in timing.items()})
pd.DataFrame(board).T.sort_values("PR_AUC", ascending=False).round(3)

**✔ Checkpoint.** Nyström trains on 254k rows in ≈ 15 s with PR-AUC ≈ 0.94 — RBF-like accuracy on 7× more data. All four models sit within ≈ 0.015 PR-AUC of the random forest, which remains (narrowly) the best.

## Step 5 · SVR for loss severity on defaulted loans

**Why this matters.** IFRS 9 expected loss = PD × LGD × EAD. Given a default, how much will we lose? This is **regression** on the ~20k defaulted loans. **SVR** with an RBF kernel can capture non-linear effects; ε defines the error we are willing to ignore. We predict **log loss** and compare with linear regression.

#### ✏️ What to write in the next cell
- **Blank 7** *(inline)* — log-scale target: losses are right-skewed
- **Blank 8** *(inline)* — width of the no-penalty tube (log units)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
loans = pd.read_parquet(CKPT / "loan_features.parquet")
d = loans[loans.default_12m == 1].copy()
d["log_loss"] = ____  # ✏️ B7
LGD_NUM = ["log_loan_amount", "tenure_months", "interest_rate", "log_salary", "debt_burden_ratio", "aecb_score"]
lgd_prep = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), LGD_NUM),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), ["product"])])
d_tr, d_te = d[d.split != "test"], d[d.split == "test"]
lin_r = Pipeline([("prep", lgd_prep), ("m", LinearRegression())]).fit(d_tr, d_tr.log_loss)
smp = d_tr.sample(6_000, random_state=RNG)
svr_gs = GridSearchCV(Pipeline([("prep", lgd_prep), ("m", SVR(kernel="rbf"))]),
                      {"m__C": [1, 10], "m__epsilon": ____, "m__gamma": ["scale", 0.05]},  # ✏️ B8
                      cv=3, scoring="neg_mean_absolute_error", n_jobs=-1).fit(smp, smp.log_loss)
print("best SVR:", svr_gs.best_params_)
res = {}
for name, m in [("linear regression", lin_r), ("SVR (tuned, 6k sample)", svr_gs.best_estimator_)]:
    pred = np.expm1(m.predict(d_te))
    res[name] = {"MAE_AED": mean_absolute_error(d_te.loss_amount_aed, pred), "MAPE": mean_absolute_percentage_error(d_te.loss_amount_aed, pred)}
pd.DataFrame(res).T.style.format({"MAE_AED": "{:,.0f}", "MAPE": "{:.1%}"})

**✔ Checkpoint.** ≈ 22k defaults; both models land at ≈ 45% MAPE and ≈ AED 16k MAE — loss severity is mostly noise (recovery outcomes are random given what we know at origination). SVR adds nothing over the linear model.

## Step 6 · Record the bake-off

**Why this matters.** Model decisions need evidence. We store accuracy **and** timing so the model inventory shows why the SVM was or was not adopted.

In [ ]:
out = {"fraud_validation": {k: {m: float(v) for m, v in r.items()} for k, r in board.items()},
       "timing_seconds": {k: round(float(v), 2) for k, v in timing.items()},
       "rbf_best_params": {k: (float(v) if not isinstance(v, str) else v) for k, v in best.items()},
       "lgd_oot": {k: {m: float(v) for m, v in r.items()} for k, r in res.items()}}
(CKPT / "svm_results.json").write_text(json.dumps(out, indent=2)); print(json.dumps(out["timing_seconds"], indent=2))

**✔ Checkpoint.** JSON written with four fraud models, timings and two LGD models.

## ✅ Lab complete
You can explain — with numbers — when an SVM is the right tool and when it isn’t. **Next:** neural networks.